# AI Lab 2: Oodles and Oodles of Models
**Course:** DS 7331 – Artificial Intelligence I 
**Team Members:** Johnny Vogt, Drew Nunnally, Devin Streeter, Mike Flores  
**Date:** 3/3/2026

# Pre-Processing and Cleaning 
Taken from Lab 1

In [47]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
from matplotlib import pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats

# Display utilities
from IPython.display import display

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import ShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn import metrics as mt
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix

# Import dataset
df = pd.read_csv("C:/Users/Owner/git/AI-ML-DS-7331/Lab1/PhiUSIIL_Phishing_URL_Dataset.csv")

# Sanity-Check the dataset
rows, cols = df.shape
print(f"The dataset contains {rows:,} rows and {cols} columns.")

# Check for duplicate rows
print("Duplicate Rows")
# Counts ONLY exact row level duplicates
display(df.duplicated().sum())

print("Duplicate URLs (only)")
# Many URLs appear more than once, but with small differences in feature values
# These are NOT full row duplicates, only URL-level duplicates
# Count duplicate values based only on the URL column
# It is safe to just drop all duplicates
display(df.duplicated(subset=["URL"]).sum())

# Drop the duplicate URLs code
# Identify duplicate URLs
dup_url_mask = df.duplicated(subset=["URL"], keep=False)

# Remove duplicate URLs (keep the first occurrence)
df_dedup = df.drop_duplicates(subset=["URL"], keep="first")
df_dedup = df_dedup.drop(columns=["FILENAME", "URL", "Domain", "Title", "TLD", "URLSimilarityIndex"], axis =1)

# New column
df_dedup['HighRiskPay'] = np.where(
    (df_dedup['Pay'] == 1 ) & (df_dedup['label'] == 0),
    1,
    0
)

rows, cols = df_dedup.shape
print(f"After dropping the duplicate URLs, the dataset contains {rows:,} rows and {cols} columns.")

# Test/Train Split
feature_cols = df_dedup.columns.drop(['label', 'HighRiskPay'])
X = df_dedup[feature_cols].values

y_label = df_dedup['label'].values
y_HighRiskPay = df_dedup['HighRiskPay'].values

#Setting up the Cross Validation using ShuffleSplit 
num_cv_iterations = 5
num_instances = len(y_label)

cv_label = StratifiedShuffleSplit(n_splits=num_cv_iterations, test_size=0.2)
cv_HighRiskPay = StratifiedShuffleSplit(n_splits=num_cv_iterations, test_size=0.2)
                         
print(cv_label)
print(cv_HighRiskPay)
print("X shape:", X.shape)
print("y_label shape:", y_label.shape)
print("y_HighRiskPay shape:", y_HighRiskPay.shape )

# Sanity check: HighRiskPay distribution
print("HighRiskPay value counts:")
print(df_dedup['HighRiskPay'].value_counts())

print("\nHighRiskPay normalized (class proportions):")
print(df_dedup['HighRiskPay'].value_counts(normalize=True))

# Extra sanity checks
print("\nUnique values in HighRiskPay:", df_dedup['HighRiskPay'].unique())
print("Any missing in HighRiskPay:", df_dedup['HighRiskPay'].isna().sum())

The dataset contains 235,795 rows and 56 columns.
Duplicate Rows


np.int64(0)

Duplicate URLs (only)


np.int64(425)

After dropping the duplicate URLs, the dataset contains 235,370 rows and 51 columns.
StratifiedShuffleSplit(n_splits=5, random_state=None, test_size=0.2,
            train_size=None)
StratifiedShuffleSplit(n_splits=5, random_state=None, test_size=0.2,
            train_size=None)
X shape: (235370, 49)
y_label shape: (235370,)
y_HighRiskPay shape: (235370,)
HighRiskPay value counts:
HighRiskPay
0    229338
1      6032
Name: count, dtype: int64

HighRiskPay normalized (class proportions):
HighRiskPay
0    0.974372
1    0.025628
Name: proportion, dtype: float64

Unique values in HighRiskPay: [0 1]
Any missing in HighRiskPay: 0


In [63]:
# Create Reusable CVHelper Function to run CV and print metrics
def CVHelper(clf, X, y, cv_object, scale=False, model_name="Model"):
    iter_num = 0
    accs, precs, recs, f1s = [], [], [], []

    for train_idx, test_idx in cv_object.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # only scale if scale=True is passed
        if scale:
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec  = recall_score(y_test, y_pred, zero_division=0)
        f1   = f1_score(y_test, y_pred, zero_division=0)
        conf = confusion_matrix(y_test, y_pred)

        accs.append(acc)
        precs.append(prec)
        recs.append(rec)
        f1s.append(f1)

        print(f"Iteration {iter_num+1} - {model_name} Metrics:")
        print(f"Accuracy: {acc:.4f}")
        print(f"Precision: {prec:.4f}")
        print(f"Recall: {rec:.4f}")
        print(f"F1-Score: {f1:.4f}")
        print(f"Confusion Matrix:\n{conf}\n")
        iter_num += 1

    avg_results = {
        "Model": model_name,
        "Accuracy": np.mean(accs),
        "Precision": np.mean(precs),
        "Recall": np.mean(recs),
        "F1-Score": np.mean(f1s),
        "F1_all": f1s,        # per-split F1 values
        "Recall_all": recs    # per-split Recall values
    }

    print(f"Average {model_name} Metrics over {iter_num} iterations:")
    print(f"Average Accuracy: {avg_results['Accuracy']:.4f}")
    print(f"Average Precision: {avg_results['Precision']:.4f}")
    print(f"Average Recall: {avg_results['Recall']:.4f}")
    print(f"Average F1-Score: {avg_results['F1-Score']:.4f}")

    return avg_results



# Models - High Risk Pay
1. Logistic Regression 
2. Naive Bayes
3. KNN

# Logistic Regression

In [64]:
# log reg for hrp

lr_hrp = LogisticRegression(C=1, class_weight='balanced', solver='lbfgs')

lr_results = CVHelper(
    clf=lr_hrp,
    X=X,
    y=y_HighRiskPay,
    cv_object=cv_HighRiskPay,
    scale=True,   # turn scaling on
    model_name="Logistic Regression for HighRiskPay"
)

Iteration 1 - Logistic Regression for HighRiskPay Metrics:
Accuracy: 0.9993
Precision: 0.9757
Recall: 0.9992
F1-Score: 0.9873
Confusion Matrix:
[[45838    30]
 [    1  1205]]

Iteration 2 - Logistic Regression for HighRiskPay Metrics:
Accuracy: 0.9995
Precision: 0.9821
Recall: 0.9983
F1-Score: 0.9901
Confusion Matrix:
[[45846    22]
 [    2  1204]]

Iteration 3 - Logistic Regression for HighRiskPay Metrics:
Accuracy: 0.9997
Precision: 0.9885
Recall: 0.9983
F1-Score: 0.9934
Confusion Matrix:
[[45854    14]
 [    2  1204]]

Iteration 4 - Logistic Regression for HighRiskPay Metrics:
Accuracy: 0.9996
Precision: 0.9885
Recall: 0.9975
F1-Score: 0.9930
Confusion Matrix:
[[45854    14]
 [    3  1203]]

Iteration 5 - Logistic Regression for HighRiskPay Metrics:
Accuracy: 0.9996
Precision: 0.9853
Recall: 1.0000
F1-Score: 0.9926
Confusion Matrix:
[[45850    18]
 [    0  1206]]

Average Logistic Regression for HighRiskPay Metrics over 5 iterations:
Average Accuracy: 0.9995
Average Precision: 0.984

# Naive Bayes

In [65]:
from sklearn.naive_bayes import GaussianNB

nb_hrp  = GaussianNB()

nb_results = CVHelper(
    clf=nb_hrp,
    X=X,
    y=y_HighRiskPay,
    cv_object=cv_HighRiskPay,
    scale=True,   # Needed for NB since the features are on different scales (e.g. URLLength:"1521345" vs. HasPay:"0,1")
    model_name="Naive Bayes for HighRiskPay"
)

Iteration 1 - Naive Bayes for HighRiskPay Metrics:
Accuracy: 0.9974
Precision: 0.9593
Recall: 0.9370
F1-Score: 0.9480
Confusion Matrix:
[[45820    48]
 [   76  1130]]

Iteration 2 - Naive Bayes for HighRiskPay Metrics:
Accuracy: 0.9975
Precision: 0.9578
Recall: 0.9420
F1-Score: 0.9498
Confusion Matrix:
[[45818    50]
 [   70  1136]]

Iteration 3 - Naive Bayes for HighRiskPay Metrics:
Accuracy: 0.9971
Precision: 0.9693
Recall: 0.9154
F1-Score: 0.9416
Confusion Matrix:
[[45833    35]
 [  102  1104]]

Iteration 4 - Naive Bayes for HighRiskPay Metrics:
Accuracy: 0.9976
Precision: 0.9505
Recall: 0.9544
F1-Score: 0.9524
Confusion Matrix:
[[45808    60]
 [   55  1151]]

Iteration 5 - Naive Bayes for HighRiskPay Metrics:
Accuracy: 0.9969
Precision: 0.9381
Recall: 0.9420
F1-Score: 0.9400
Confusion Matrix:
[[45793    75]
 [   70  1136]]

Average Naive Bayes for HighRiskPay Metrics over 5 iterations:
Average Accuracy: 0.9973
Average Precision: 0.9550
Average Recall: 0.9381
Average F1-Score: 0.946

# K Nearest Neighbor

In [66]:
from sklearn.neighbors import KNeighborsClassifier

knn_results_list = []

for k in (3, 5, 7, 9, 11, 13, 15):  # 3 -15 Odds Only to avoid ties in KNN
    knn_hrp = KNeighborsClassifier(
        n_neighbors=k,
        weights='distance',
        metric='minkowski',
        p=2
    )

    print(f"\n======== Running KNN (k={k}) for HighRiskPay ========")
    result = CVHelper(
        clf=knn_hrp,
        X=X,
        y=y_HighRiskPay,
        cv_object=cv_HighRiskPay,
        scale=True,
        model_name=f"KNN (k={k}) for HighRiskPay"
    )
    knn_results_list.append(result)

    knn_df = pd.DataFrame(knn_results_list)
display(knn_df)

# Pick best model by highest F1-Score
best_idx = knn_df['F1-Score'].idxmax()
best_row = knn_df.loc[best_idx]
knn_results = knn_df.loc[best_idx].to_dict()

print("\nBest KNN configuration based on F1-Score:")
print(best_row)


======== Running KNN (k=3) for HighRiskPay ========
Iteration 1 - KNN (k=3) for HighRiskPay Metrics:
Accuracy: 0.9969
Precision: 0.9853
Recall: 0.8905
F1-Score: 0.9355
Confusion Matrix:
[[45852    16]
 [  132  1074]]

Iteration 2 - KNN (k=3) for HighRiskPay Metrics:
Accuracy: 0.9974
Precision: 0.9901
Recall: 0.9088
F1-Score: 0.9477
Confusion Matrix:
[[45857    11]
 [  110  1096]]

Iteration 3 - KNN (k=3) for HighRiskPay Metrics:
Accuracy: 0.9969
Precision: 0.9810
Recall: 0.8980
F1-Score: 0.9377
Confusion Matrix:
[[45847    21]
 [  123  1083]]

Iteration 4 - KNN (k=3) for HighRiskPay Metrics:
Accuracy: 0.9968
Precision: 0.9826
Recall: 0.8905
F1-Score: 0.9343
Confusion Matrix:
[[45849    19]
 [  132  1074]]

Iteration 5 - KNN (k=3) for HighRiskPay Metrics:
Accuracy: 0.9971
Precision: 0.9872
Recall: 0.8980
F1-Score: 0.9405
Confusion Matrix:
[[45854    14]
 [  123  1083]]

Average KNN (k=3) for HighRiskPay Metrics over 5 iterations:
Average Accuracy: 0.9970
Average Precision: 0.9852
Avera

,Model,Accuracy,Precision,Recall,F1-Score,F1_all,Recall_all
0,KNN (k=3) for HighRiskPay,0.997022,0.985243,0.897181,0.939144,"[0.9355400696864111, 0.9476869865974924, 0.937...","[0.8905472636815921, 0.9087893864013267, 0.898..."
1,KNN (k=5) for HighRiskPay,0.996886,0.990561,0.886899,0.935848,"[0.9347921225382932, 0.936897458369851, 0.9431...","[0.8855721393034826, 0.8864013266998342, 0.901..."
2,KNN (k=7) for HighRiskPay,0.996444,0.992987,0.867330,0.925887,"[0.9238938053097345, 0.9205357142857142, 0.921...","[0.8656716417910447, 0.8548922056384743, 0.862..."
3,KNN (k=9) for HighRiskPay,0.996036,0.989046,0.854726,0.916946,"[0.9155555555555556, 0.9327472527472528, 0.915...","[0.8540630182421227, 0.8797678275290216, 0.852..."
4,KNN (k=11) for HighRiskPay,0.995955,0.991683,0.849254,0.914949,"[0.9125952487673689, 0.9159626500666963, 0.912...","[0.8441127694859039, 0.8540630182421227, 0.844..."
5,KNN (k=13) for HighRiskPay,0.995764,0.990076,0.843118,0.910688,"[0.9072164948453608, 0.9097408400357462, 0.905...","[0.8391376451077943, 0.8441127694859039, 0.831..."
6,KNN (k=15) for HighRiskPay,0.995382,0.991442,0.826866,0.901692,"[0.9058134294727355, 0.9019430637144148, 0.904...","[0.8333333333333334, 0.8275290215588723, 0.831..."



Best KNN configuration based on F1-Score:
Model                                 KNN (k=3) for HighRiskPay
Accuracy                                               0.997022
Precision                                              0.985243
Recall                                                 0.897181
F1-Score                                               0.939144
F1_all        [0.9355400696864111, 0.9476869865974924, 0.937...
Recall_all    [0.8905472636815921, 0.9087893864013267, 0.898...
Name: 0, dtype: object


# Performance Metrics Comparison Table - Predict fradulent websites with payment capabilities

In [67]:
#Buidling Table for all the models and their metrics
all_results = [lr_results, nb_results, knn_results]
results_df = pd.DataFrame(all_results)
display(results_df)


,Model,Accuracy,Precision,Recall,F1-Score,F1_all,Recall_all
0,Logistic Regression for HighRiskPay,0.999550,0.984012,0.998673,0.991282,"[0.9873002867677182, 0.9901315789473685, 0.993...","[0.9991708126036484, 0.9983416252072969, 0.998..."
1,Naive Bayes for HighRiskPay,0.997277,0.954978,0.938143,0.946365,"[0.947986577181208, 0.9498327759197325, 0.9415...","[0.9369817578772802, 0.9419568822553898, 0.915..."
2,KNN (k=3) for HighRiskPay,0.997022,0.985243,0.897181,0.939144,"[0.9355400696864111, 0.9476869865974924, 0.937...","[0.8905472636815921, 0.9087893864013267, 0.898..."


In [70]:
# 95% CI for F1 for All Models
import numpy as np

def print_ci(name, values):
    vals = np.array(values)
    mean = vals.mean()
    se = vals.std(ddof=1) / np.sqrt(len(vals))
    ci_low = mean - 1.96 * se
    ci_high = mean + 1.96 * se
    print(f"{name} F1 95% CI: mean={mean:.4f}, CI=({ci_low:.4f}, {ci_high:.4f})")

print_ci("Logistic Regression", lr_results["F1_all"])
print_ci("Naive Bayes", nb_results["F1_all"])
print_ci("KNN", knn_results["F1_all"])


Logistic Regression F1 95% CI: mean=0.9913, CI=(0.9890, 0.9935)
Naive Bayes F1 95% CI: mean=0.9464, CI=(0.9417, 0.9511)
KNN F1 95% CI: mean=0.9391, CI=(0.9345, 0.9438)
